<a href="https://colab.research.google.com/github/AmanuelDaget/codealpha_tasks/blob/main/Emotion_Recognition_from_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install dependencies**

In [ ]:
# pip install librosa soundfile scikit-learn tensorflow numpy matplotlib seaborn

**Import Libraries**

In [7]:
import os
import numpy as np
import librosa
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, Dense, Dropout,
                                      BatchNormalization, Input, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from google.colab import drive

**Mount drive**

In [8]:
drive.mount('/content/drive')

Mounted at /content/drive


**Set your dataset path here**

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/CodeAlpha/Tess_Dataset"
SAVE_PATH    = "/content/drive/MyDrive/Colab Notebooks/CodeAlpha/Emotion Recognition/tess_model"
os.makedirs(SAVE_PATH, exist_ok=True)


**TESS LABEL PARSING**

In [9]:
VALID_EMOTIONS = {'angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad'}
EMOTION_RENAME = {'ps': 'pleasant_surprise'}   # rename if you prefer

def parse_tess_label(filepath):
    """Extract emotion from TESS filename."""
    fname  = os.path.splitext(os.path.basename(filepath))[0]  # drop .wav
    parts  = fname.lower().split('_')
    emotion = parts[-1]                                         # last token
    if emotion not in VALID_EMOTIONS:
        return None
    return EMOTION_RENAME.get(emotion, emotion)

In [ ]:
MAX_LEN    = 200   # time frames (≈ 3 s at 22050 Hz with default hop)
N_FEATURES = 133   # 40 MFCC + 40 Δ + 40 ΔΔ + 12 Chroma + 1 ZCR + 1 RMS - wait, see stack below

def extract_features(y, sr, max_len=MAX_LEN):
    """
    Input : raw waveform array + sample rate
    Output: (max_len, 133) float32 array

    Feature stack (all computed on the same hop_length so shapes align):
        MFCC        40 coeffs  — timbral texture
        Δ  MFCC     40 coeffs  — rate of change
        ΔΔ MFCC     40 coeffs  — acceleration
        Chroma      12 coeffs  — pitch-class energy
        Mel (dB)    40 coeffs  — perceptual energy bands   → wait, 40+40+40+12 = 132 + zcr+rms = 134
        ZCR          1         — voiced/unvoiced boundary
        RMS          1         — loudness / energy
                   ────────
                   174 total  (see note below)
    """
    # Pre-processing
    y, _ = librosa.effects.trim(y, top_db=20)                  # strip silence
    y    = np.append(y[0], y[1:] - 0.97 * y[:-1])             # pre-emphasis

    hop = 512
    # Feature extraction
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40, hop_length=hop)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12, hop_length=hop)
    mel    = librosa.power_to_db(
                 librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40, hop_length=hop),
                 ref=np.max)
    zcr    = librosa.feature.zero_crossing_rate(y, hop_length=hop)
    rms    = librosa.feature.rms(y=y, hop_length=hop)

    # Stack → (T, n_features)
    features = np.vstack([mfcc, delta, delta2, chroma, mel, zcr, rms]).T.astype(np.float32)

    # Pad / truncate to fixed length
    T = features.shape[0]
    if T < max_len:
        features = np.vstack([features, np.zeros((max_len - T, features.shape[1]), dtype=np.float32)])
    else:
        features = features[:max_len]

    return features   # (max_len, n_features)


In [ ]:
# ─────────────────────────────────────────────
def augment(y, sr):
    """Yields augmented copies of the waveform (3 variants)."""
    # Gaussian noise
    yield y + np.random.normal(0, 0.004, len(y)).astype(np.float32)
    # Time-stretch (slow)
    try: yield librosa.effects.time_stretch(y, rate=0.9)
    except: pass
    # Pitch shift (+2 semitones)
    try: yield librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
    except: pass

In [ ]:
def load_tess(dataset_path, use_augment=True):
    X, labels = [], []
    skipped   = 0

    for root, _, files in os.walk(dataset_path):
        for fname in sorted(files):
            if not fname.lower().endswith('.wav'):
                continue

            fpath   = os.path.join(root, fname)
            emotion = parse_tess_label(fpath)
            if emotion is None:
                skipped += 1
                continue

            try:
                y, sr = librosa.load(fpath, sr=22050, duration=3.5, offset=0.0)
            except Exception as e:
                print(f"  ⚠ Cannot load {fname}: {e}")
                skipped += 1
                continue

            # Original
            feat = extract_features(y, sr)
            X.append(feat); labels.append(emotion)

            # Augmented copies
            if use_augment:
                for y_aug in augment(y, sr):
                    X.append(extract_features(y_aug, sr))
                    labels.append(emotion)

    print(f"✅  {len(labels)} samples loaded  |  {skipped} skipped")

    # Class distribution
    unique, counts = np.unique(labels, return_counts=True)
    print("📊  Class distribution:")
    for e, c in zip(unique, counts):
        print(f"     {e:<20} {c}")

    return np.array(X, dtype=np.float32), np.array(labels)

